In [28]:
from backend.app.services.data_services import (
    get_available_indicators,
    get_available_strategies,
    get_strategies_metadata,
    get_available_tickers,
    _read_csv,
    fetch_data_to_df,
    get_indicators_metadata,
    _load_and_resample_data
)
from backend.app.services.cache_service import set_data, delete_data, get_key_list
from backend.app.core.Strategies import STRATEGY_REGISTRY

import asyncio
import pandas as pd
import vectorbt as vbt

In [10]:
ticker_name = 'RELIANCE INDUSTRIES LTD'
df = await _read_csv(ticker_name=ticker_name)
await set_data(df=df, ticker= ticker_name)

{'message': 'Data loaded successfully with key: data:RELIANCE INDUSTRIES LTD'}

In [22]:
# get_key_list()
# await delete_data("NIFTY 50")
# get_available_tickers()
get_available_strategies()

['EMA Crossover',
 'SMA Crossover',
 'RSI',
 'MACD',
 'Bollinger Bands',
 'VWAP',
 'ADX Trend',
 'Stochastic Oscillator',
 'CCI',
 'ROC',
 'Williams %R',
 'ATR Breakout',
 'Donchian Channel',
 'Momentum',
 'EMA + RSI',
 'Mean Reversion',
 'Price Channel',
 'RSI Divergence',
 'EMA Pullback',
 'SuperTrend']

In [31]:
resolution = "15m"
start_date = "01/12/2024 09:15:00"
end_date = "01/02/2025 09:15:00"

df = await _load_and_resample_data(ticker_name, resolution, start_date, end_date)
df.shape

d:\OneDrive - iitgn.ac.in\Desktop\HedgeOne-Quant\backend\app\services\data_services.py:54: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.



(1101, 6)

In [63]:
strategy_fn = STRATEGY_REGISTRY["EMA Crossover"]
params = {"short_window":4, "long_window":9}

result = strategy_fn(df, **params)

portfolio = await asyncio.to_thread(
    vbt.Portfolio.from_signals,
    df["close"],
    result["entries"],
    result["exits"],
    freq=pd.Timedelta(resolution),
    init_cash=100000,
    fees=0,
    slippage=0
)

In [64]:
portfolio.stats(group_by=True)

Start                                                 0
End                                                1100
Period                                 11 days 11:15:00
Start Value                                    100000.0
End Value                                 100296.129454
Total Return [%]                               0.296129
Benchmark Return [%]                          -0.891461
Max Gross Exposure [%]                            100.0
Total Fees Paid                                     0.0
Max Drawdown [%]                               4.095371
Max Drawdown Duration                   4 days 04:30:00
Total Trades                                         50
Total Closed Trades                                  49
Total Open Trades                                     1
Open Trade PnL                               134.530077
Win Rate [%]                                  28.571429
Best Trade [%]                                 3.987521
Worst Trade [%]                               -1

In [65]:
portfolio.trades.records

,id,col,size,entry_idx,entry_price,entry_fees,exit_idx,exit_price,exit_fees,pnl,return,direction,status,parent_id
0,0,0,75.520145,48,1324.15,0.0,50,1319.00,0.0,-388.928747,-0.003889,0,1,0
1,1,0,75.856582,68,1313.15,0.0,69,1311.40,0.0,-132.749019,-0.001333,0,1,1
2,2,0,75.735304,75,1313.50,0.0,78,1307.95,0.0,-420.330939,-0.004225,0,1,2
3,3,0,75.561990,79,1310.95,0.0,83,1309.80,0.0,-86.896289,-0.000877,0,1,3
4,4,0,75.438161,85,1311.95,0.0,100,1317.30,0.0,403.594160,0.004078,0,1,4
5,5,0,77.277257,174,1285.95,0.0,179,1283.65,0.0,-177.737692,-0.001789,0,1,5
6,6,0,77.121051,184,1286.25,0.0,187,1283.15,0.0,-239.075257,-0.002410,0,1,6
7,7,0,78.816356,236,1255.55,0.0,256,1271.30,0.0,1241.357613,0.012544,0,1,7
8,8,0,80.012165,300,1252.30,0.0,311,1252.05,0.0,-20.003041,-0.000200,0,1,8
9,9,0,79.910047,318,1253.65,0.0,325,1235.50,0.0,-1450.367358,-0.014478,0,1,9


In [66]:
portfolio.value()

0       100000.000000
1       100000.000000
2       100000.000000
3       100000.000000
4       100000.000000
            ...      
1096    100161.599377
1097    100161.599377
1098    100161.599377
1099    100185.339978
1100    100296.129454
Name: close, Length: 1101, dtype: float64

In [67]:
p = portfolio

In [68]:
p.orders.records

,id,col,idx,size,price,fees,side
0,0,0,48,75.520145,1324.15,0.0,0
1,1,0,50,75.520145,1319.00,0.0,1
2,2,0,68,75.856582,1313.15,0.0,0
3,3,0,69,75.856582,1311.40,0.0,1
4,4,0,75,75.735304,1313.50,0.0,0
...,...,...,...,...,...,...,...
94,94,0,1052,79.916869,1240.10,0.0,0
95,95,0,1068,79.916869,1247.00,0.0,1
96,96,0,1073,79.569113,1252.45,0.0,0
97,97,0,1094,79.569113,1258.80,0.0,1


In [69]:
d = pd.DataFrame(p.positions.records)
d.columns

Index(['id', 'col', 'size', 'entry_idx', 'entry_price', 'entry_fees',
       'exit_idx', 'exit_price', 'exit_fees', 'pnl', 'return', 'direction',
       'status', 'parent_id'],
      dtype='object')

In [70]:
d.head()

,id,col,size,entry_idx,entry_price,entry_fees,exit_idx,exit_price,exit_fees,pnl,return,direction,status,parent_id
0,0,0,75.520145,48,1324.15,0.0,50,1319.00,0.0,-388.928747,-0.003889,0,1,0
1,1,0,75.856582,68,1313.15,0.0,69,1311.40,0.0,-132.749019,-0.001333,0,1,1
2,2,0,75.735304,75,1313.50,0.0,78,1307.95,0.0,-420.330939,-0.004225,0,1,2
3,3,0,75.561990,79,1310.95,0.0,83,1309.80,0.0,-86.896289,-0.000877,0,1,3
4,4,0,75.438161,85,1311.95,0.0,100,1317.30,0.0,403.594160,0.004078,0,1,4


In [71]:
p.plot()

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': '#1f77b4'},
              'name': 'Close',
              'showlegend': True,
              'type': 'scatter',
              'uid': '49f44a5b-19c5-4f22-be38-1f1f8927e250',
              'x': {'bdata': ('AAABAAIAAwAEAAUABgAHAAgACQAKAA' ... 'RCBEMERARFBEYERwRIBEkESgRLBEwE'),
                    'dtype': 'i2'},
              'xaxis': 'x',
              'y': {'bdata': ('MzMzMzP7k0BmZmZmZhSUQAAAAAAAGJ' ... 'zMzMzGk0AAAAAAAMiTQJqZmZmZzZNA'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': ('AAAAAAAAAAAboD8OSuFSQAAAAAAAAA' ... 'AAAACAWEAbCZNnqchTQAAAAAAAAAAA'),
                             'dtype': 'f8',
                             'shape': '50, 3'},
              'hovertemplate': ('Order Id: %{customdata[0]}<br>' ... '<br>Fees: %{customdata[2]:.6f}'),
              'legendgroup': '1',
              'marker': {'color': '#37B13F',
                         'lin

In [54]:
dir(p)

['__annotations__',
 '__cached_asset_flow',
 '__cached_asset_value',
 '__cached_assets',
 '__cached_benchmark_returns',
 '__cached_benchmark_value',
 '__cached_cash',
 '__cached_cash_flow',
 '__cached_exit_trades',
 '__cached_final_value',
 '__cached_get_drawdowns',
 '__cached_get_exit_trades',
 '__cached_get_filled_close',
 '__cached_get_init_cash',
 '__cached_get_orders',
 '__cached_get_positions',
 '__cached_get_returns_acc',
 '__cached_get_trades',
 '__cached_gross_exposure',
 '__cached_orders',
 '__cached_positions',
 '__cached_returns',
 '__cached_total_profit',
 '__cached_total_return',
 '__cached_trades',
 '__cached_value',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclas